## Importando as Bibliotecas

In [16]:
# ================================
# Manipulação de dados
# ================================
import pandas as pd
import numpy as np

# ================================
# Visualização de dados
# ================================
import matplotlib.pyplot as plt
import seaborn as sns

# ================================
# Salvamento de modelos
# ================================
import joblib

# ================================
# Divisão dos dados
# ================================
from sklearn.model_selection import train_test_split

# ================================
# Pré-processamento
# ================================
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# ================================
# Modelos de Machine Learning
# ================================
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import xgboost as xgb

# ================================
# Avaliação de modelos
# ================================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# ================================
# Validação e otimização
# ================================
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_validate, StratifiedKFold, cross_val_score

import time

print("Bibliotecas Importadas!!")

Bibliotecas Importadas!!


# 2. Leitura da base tratada

In [2]:
dados = pd.read_parquet("../data/gold/dataset_modelagem.parquet")
dados.head()

,ano,rede,sigla_uf,regiao,pib_per_capita,populacao_2022,area_km2,densidade_demografica,qtd_escolas_censo,prop_escolas_rurais,...,prop_escolas_biblioteca_leitura,prop_escolas_lab_informatica,prop_escolas_internet,prop_escolas_internet_aprendizagem,prop_escolas_banda_larga,prop_escolas_rampas_acessibilidade,media_alunos_turma_fund_ai,media_alunos_docente_fund_ai,media_prop_salas_climatizadas,alfabetizado
0,2023,Municipal,AM,Norte,17433.75,101883,7336.579,13.89,165,0.872727,...,0.132353,0.029412,0.705882,0.411765,0.541667,0.308824,19.188693,16.360779,0.642714,Não
1,2023,Municipal,AM,Norte,54985.63,2063689,11401.092,181.01,532,0.167293,...,0.531062,0.529058,0.969940,0.665331,0.878099,0.492986,25.931574,28.238906,0.920097,Não
2,2023,Municipal,AM,Norte,11569.04,20718,17472.779,1.19,97,0.969072,...,0.044944,0.000000,0.292135,0.224719,0.884615,0.056180,15.716064,14.607079,0.088443,Não
3,2023,Municipal,TO,Norte,28536.72,3334,2167.201,1.54,7,0.571429,...,0.200000,0.200000,1.000000,0.200000,0.400000,0.600000,17.472222,17.472222,0.200000,Não
4,2023,Municipal,MA,Nordeste,9325.56,25322,940.489,26.92,63,0.904762,...,0.153846,0.000000,0.948718,0.179487,1.000000,0.384615,25.611508,16.573413,0.198077,Não


# 3. Validação rápida dos Dados

In [3]:
print(f"Total de linhas: {dados.shape[0]}")
print(f"Total de colunas: {dados.shape[1]}")

Total de linhas: 57712
Total de colunas: 22


In [4]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57712 entries, 0 to 57711
Data columns (total 22 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   ano                                 57712 non-null  Int64  
 1   rede                                57712 non-null  string 
 2   sigla_uf                            57712 non-null  object 
 3   regiao                              57712 non-null  object 
 4   pib_per_capita                      57712 non-null  float64
 5   populacao_2022                      57712 non-null  int64  
 6   area_km2                            57712 non-null  float64
 7   densidade_demografica               57712 non-null  float64
 8   qtd_escolas_censo                   57712 non-null  Int64  
 9   prop_escolas_rurais                 57712 non-null  float64
 10  prop_escolas_agua_potavel           57712 non-null  float64
 11  prop_escolas_esgoto_rede            57712

# 4. Definição da variável alvo

In [6]:
target = {
    'Não':0,
    'Sim': 1
}

dados['alfabetizado'] = dados['alfabetizado'].map(target)

In [8]:
dados['alfabetizado'].value_counts()

alfabetizado
0    29468
1    28244
Name: count, dtype: int64

# 5. Divisão dos dados em treino, validação e teste

O dataset foi dividido em três conjuntos independentes:

- **64% para treinamento:** utilizado para ajuste dos parâmetros do modelo;
- **16% para validação:** utilizado para comparação entre modelos e seleção de hiperparâmetros;
- **20% para teste:** reservado para a avaliação final do modelo selecionado.

A divisão foi realizada de forma estratificada pela variável alvo `alfabetizado`, preservando aproximadamente a mesma proporção entre as classes em todos os conjuntos.

Também foi definido `random_state=42` para garantir reprodutibilidade dos experimentos.

O conjunto de teste será mantido isolado durante as etapas de treinamento e seleção dos modelos, sendo utilizado apenas na avaliação final.

In [9]:
X = dados.drop('alfabetizado', axis=1)
y = dados['alfabetizado']

In [13]:
# 1º Divisão -> treino + validação e Teste
X_temp, X_teste, y_temp, y_teste = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# 2º Divisão -> treino e validação
X_treino, X_validacao, y_treino, y_validacao = train_test_split(X_temp, y_temp, stratify=y_temp, test_size=0.2, random_state=42)

In [14]:
print(f"Treino -> {X_treino.shape[0]}")
print(f"Validação -> {X_validacao.shape[0]}")
print(f"Teste -> {X_teste.shape[0]}")

Treino -> 36935
Validação -> 9234
Teste -> 11543


# 6. Pré-processamento dos Dados

In [23]:
col_numericas = ['ano','pib_per_capita','populacao_2022','area_km2','densidade_demografica',
                 'qtd_escolas_censo','prop_escolas_rurais','prop_escolas_agua_potavel','prop_escolas_esgoto_rede','prop_escolas_biblioteca_leitura',
                 'prop_escolas_lab_informatica','prop_escolas_internet','prop_escolas_internet_aprendizagem','prop_escolas_banda_larga',
                 'prop_escolas_rampas_acessibilidade','media_alunos_turma_fund_ai','media_alunos_docente_fund_ai','media_prop_salas_climatizadas']

col_categoricas = ['rede', 'sigla_uf', 'regiao']

In [24]:
pipeline_numerico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [25]:
pipeline_categorico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop='first', handle_unknown="ignore",sparse_output=False))
    ]
)

In [26]:
preprocessador = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, col_numericas),
        ("cat", pipeline_categorico, col_categoricas)
    ]
)

X_treino_tratado = preprocessador.fit_transform(X_treino)
X_validacao_tratado = preprocessador.transform(X_validacao)
X_teste_tratado = preprocessador.transform(X_teste)

In [28]:
num_cols = list(col_numericas)
cat_cols = preprocessador.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(col_categoricas).tolist()

X_treino = pd.DataFrame(X_treino_tratado, columns=num_cols + cat_cols)
X_Validacao = pd.DataFrame(X_validacao_tratado, columns=num_cols + cat_cols)
X_Teste = pd.DataFrame(X_teste_tratado, columns=num_cols + cat_cols)

In [29]:
X_treino.head()

,ano,pib_per_capita,populacao_2022,area_km2,densidade_demografica,qtd_escolas_censo,prop_escolas_rurais,prop_escolas_agua_potavel,prop_escolas_esgoto_rede,prop_escolas_biblioteca_leitura,...,sigla_uf_RO,sigla_uf_RS,sigla_uf_SC,sigla_uf_SE,sigla_uf_SP,sigla_uf_TO,regiao_Nordeste,regiao_Norte,regiao_Sudeste,regiao_Sul
0,0.988024,0.105924,-0.189908,-0.312104,5.419766,-0.287411,-0.959160,0.274539,1.176443,0.464790,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,-1.012122,0.242369,-0.367277,0.375782,-0.521020,-0.444908,0.864954,0.274539,-0.685777,-1.053742,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.988024,0.720633,-0.352832,-0.289006,-0.404292,-0.355759,0.140483,0.274539,-0.784832,-1.351548,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-1.012122,2.096789,-0.355933,-0.258595,-0.473706,-0.453823,-0.466939,-0.562573,0.245333,0.530813,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-1.012122,-0.844085,-0.370519,-0.268045,-0.497630,-0.438965,1.579665,0.274539,-0.840962,-0.129418,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
